### MINIMIZACAO EICONAL ATE t = 0.1 COM 3 PARAMETROS LIVRES

In [ ]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad, quad

In [ ]:
# # Load experimental data
# atlas_data = pd.read_csv('../../data/data_0_1/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)

# # Function to process data for each experiment
# def process_data(data, energy_blocks):
#     x_values = []
#     y_values = []
#     y_errors = []
    
#     for start, end in energy_blocks:
#         block = data.iloc[start:end] if end is not None else data.iloc[start:]
#         x_values.append(block[0].values)
#         y_values.append(block[1].values)
#         y_errors.append(block[2].values)
    
#     return x_values, y_values, y_errors

# # Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
# atlas_blocks = [(0, 29), (29, 58), (58, None)]
# totem_blocks = [(0, 65), (65, 118), (118, None)]

# # Process data
# x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# # Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
# x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
# x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
# x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [ ]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25

In [ ]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q
    qk_cos = np.sqrt(np.abs(q)) * np.abs(k) * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(np.abs(q)) * np.abs(k) * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 


def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  



In [ ]:
n_points = 8000  # Number of points for fixed_quad integration

def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):
    q_val = np.atleast_1d(q_val)
    results = []

    for q in q_val:                      # <- q is the correct variable
        def integrand(y, x, mg, a1, a2, m2_func, q):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            return k * (
                T_1(k, q, phi, mg, a1, a2, m2_func)   # <-- FIXED: q instead of q_val
                - T_2(k, q, phi, mg, a1, a2, m2_func)
            ) * jacobian

        def inner_integral(x):
            integral_real = fixed_quad(
                lambda y: np.real(integrand(y, x, mg, a1, a2, m2_func, q)),
                0, 1, n=n_points
            )[0]
            integral_imag = fixed_quad(
                lambda y: np.imag(integrand(y, x, mg, a1, a2, m2_func, q)),
                0, 1, n=n_points
            )[0]
            return integral_real + 1j * integral_imag

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    return np.array(results) if len(results) > 1 else results[0]


In [ ]:
lst_chi = []
lst_amp_eik = []
lst_diff_sigma = []


q_max_chi = 5.0          # limite de q na Eq. 23
b_max     = 15.0



# ── Eq. 23: χ(s,b) como integral direta em q ─────────────────────────────────
def chi(b_val, mg, a1, a2, eps, m2_func, sqrt_s):

    s_local = sqrt_s ** 2  # [CORREÇÃO 1] s local, não captura variável global

    def integrand(q_val):
        """Integrando complexo — full_int chamado uma única vez por ponto."""
        # [CORREÇÃO 2] integrando único complexo: evita chamar full_int 2x
        q2_val    = q_val ** 2
        t         = -q2_val
        diff_t    = full_int(mg, a1, a2, m2_func, q2_val, sqrt_s)
        born_amp  = amp_calculation(diff_t, s_local, eps, t)
        return (1.0 / s_local) * q_val * j0(b_val * q_val) * born_amp

 
    real_part, _ = fixed_quad(lambda q: np.real(integrand(q)), 0, q_max_chi, n=n_points)
    imag_part, _ = fixed_quad(lambda q: np.imag(integrand(q)), 0, q_max_chi, n=n_points)
    # print(f"b = {b_val:.2f}, chi_real = {real_part:.6f}, chi_imag = {imag_part:.6f}")

    return real_part + 1j * imag_part

In [ ]:
mg_eik = 0.380
eps_eik = 0.0892
a1_eik = 1.491
a2_eik = 2.77

In [ ]:
lst_b_integration = np.linspace(0, b_max, 1000)

In [ ]:
for b_val in lst_b_integration:
    chi_value = chi(b_val, mg_eik, a1_eik, a2_eik, eps_eik, m2_log, 7000)
    print(f"b = {b_val:.2f}, chi = {chi_value:.6f}")